# Build a Prompt Injection Attack Against a Basic Chat Assistant

This demo tests how malicious instructions embedded in chat content can change assistant behavior. The default path uses a deterministic mock assistant so the demo works without an API key. An optional OpenAI API cell is included for live model testing.

## 1. Load the Demo Environment

In [ ]:
from pathlib import Path
import os
import sys

CURRENT = Path.cwd().resolve()
ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
sys.path.append(str(ROOT / "src"))

from chat_injection_utils import (
    GuardedMockAssistant,
    MockVulnerableAssistant,
    build_messages,
    detect_in_messages,
    load_json,
    load_text,
    output_is_compromised,
    run_injection_suite,
    run_openai_response,
    summarize_runs,
    write_runs_csv,
)

DATA_DIR = ROOT / "data"
PROMPT_DIR = ROOT / "prompts"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

system_prompt = load_text(PROMPT_DIR / "system_prompt.md")
secure_system_prompt = load_text(PROMPT_DIR / "secure_system_prompt.md")
benign_queries = load_json(DATA_DIR / "benign_user_queries.json")
payloads = load_json(DATA_DIR / "malicious_prompt_injection_payloads.json")
history_templates = load_json(DATA_DIR / "chat_history_templates.json")

clean_history = history_templates[0]["messages"]
injected_history = history_templates[1]["messages"]
benign_query = benign_queries[0]["query"]

print(f"Loaded {len(benign_queries)} benign queries")
print(f"Loaded {len(payloads)} injection payloads")
print(f"Selected benign query: {benign_query}")

## 2. Review the System Prompt and Chat Flow

The application assembles three types of context: system instructions, optional chat history, and the current user prompt. Prompt injection becomes possible when untrusted text is placed near trusted instructions without clear isolation.

In [ ]:
print(system_prompt)

baseline_messages = build_messages(benign_query, history=clean_history)
baseline_messages

## 3. Test Baseline Behavior on a Benign Query

In [ ]:
vulnerable_assistant = MockVulnerableAssistant()
baseline_response = vulnerable_assistant.respond(system_prompt, baseline_messages)

print(baseline_response)
print("Compromised output?", output_is_compromised(baseline_response))

## 4. Inject Malicious Payloads

The first payloads are placed directly in the user prompt. Others simulate attacker-controlled text from prior conversation context.

In [ ]:
unsafe_rows = run_injection_suite(
    vulnerable_assistant,
    system_prompt,
    benign_query,
    payloads,
    clean_history=clean_history,
    injected_history=injected_history,
)

write_runs_csv(unsafe_rows, RESULTS_DIR / "unsafe_prompt_injection_results.csv")
summarize_runs(unsafe_rows)

## 5. Inspect One Direct Injection and One History Injection

In [ ]:
for row in unsafe_rows:
    if row.test_id in ["p001", "p003"]:
        print("=" * 80)
        print(f"Test: {row.test_id} | placement: {row.placement} | payload: {row.payload}")
        print(f"Detected before model call? {row.detected}")
        print(f"Compromised response? {row.compromised}")
        print(row.response)


## 6. Optional: Run One Case Against the OpenAI API

Set `USE_OPENAI = True` only when `OPENAI_API_KEY` is available. The call uses the Responses API: system instructions are passed in `instructions`, and chat messages are passed in `input`.

In [ ]:
USE_OPENAI = False
OPENAI_MODEL = "gpt-5.1-mini"

if USE_OPENAI:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before enabling USE_OPENAI.")

    sample_payload = payloads[0]
    messages = build_messages(
        benign_query,
        history=clean_history,
        injected_payload=sample_payload["payload"],
        placement=sample_payload["placement"],
    )
    live_response = run_openai_response(system_prompt, messages, model=OPENAI_MODEL)
    print(live_response)
else:
    print("Skipping live API call. Set USE_OPENAI = True to test one case against OpenAI.")

## 7. Add Simple Defensive Controls

The guarded mock assistant demonstrates three defensive ideas:

- Treat history and copied content as untrusted data.
- Detect common instruction-override patterns before the model call.
- Validate output for compromise markers.

In [ ]:
guarded_assistant = GuardedMockAssistant()
guarded_rows = run_injection_suite(
    guarded_assistant,
    secure_system_prompt,
    benign_query,
    payloads,
    clean_history=clean_history,
    injected_history=injected_history,
)

write_runs_csv(guarded_rows, RESULTS_DIR / "guarded_prompt_injection_results.csv")
summarize_runs(guarded_rows)

## 8. Compare Outcomes

In [ ]:
unsafe_compromised = sum(row.compromised for row in unsafe_rows)
guarded_compromised = sum(row.compromised for row in guarded_rows)

print(f"Unsafe compromised cases: {unsafe_compromised} / {len(unsafe_rows)}")
print(f"Guarded compromised cases: {guarded_compromised} / {len(guarded_rows)}")

for unsafe, guarded in zip(unsafe_rows, guarded_rows):
    print(
        f"{unsafe.test_id}: unsafe_compromised={unsafe.compromised}, "
        f"guarded_compromised={guarded.compromised}, detected={guarded.detected}"
    )

## Key Takeaway

Prompt injection exploits the fact that chat models interpret conversation text as possible instructions. For a basic chat assistant, all user-provided text, copied content, retrieved context, and conversation history must be handled as untrusted input.